In [0]:
# Databricks notebook source
from pyspark.sql.functions import current_timestamp, col

# Define parameters via widgets
dbutils.widgets.text("catalog", "dbr_dev", "1. Catalog Name")
dbutils.widgets.text("schema", "valeriimatviiv_bronze", "2. Schema Name")
dbutils.widgets.text("volume", "market_radar_landing", "3. Landing Volume")
dbutils.widgets.text("target_table", "raw_finnhub_news", "4. Target Delta Table")
dbutils.widgets.text("max_files_per_trigger", "50", "5. Max Files Per Batch")

# Retrieve widget values
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = dbutils.widgets.get("volume")
target_table_name = dbutils.widgets.get("target_table")
max_files_per_trigger = int(dbutils.widgets.get("max_files_per_trigger"))

base_path = f"/Volumes/{catalog}/{schema}/{volume}"
landing_path = f"{base_path}/landing/finnhub_news"
checkpoint_path = f"{base_path}/_state/checkpoints/finnhub_news"
schema_path = f"{base_path}/_state/schemas/finnhub_news"
full_table_path = f"{catalog}.{schema}.{target_table_name}"

In [0]:
# 1. Configure Auto Loader Stream
df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.maxFilesPerTrigger", max_files_per_trigger)
    .load(landing_path)
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))  # Unity Catalog compliant source file tracking
)

# 2. Write Micro-Batch Stream to Delta Bronze Table
query = (
    df_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(full_table_path)
)

query.awaitTermination()

print(f"Streaming execution completed for Delta table: {full_table_path}")

In [0]:
# # Verify total ingested records in target table
# df_target = spark.table(f"{catalog}.{schema}.{target_table_name}")

# print(f"Total rows in Delta table: {df_target.count()}")
# print("\nTarget Delta Table Schema:")
# df_target.printSchema()

# # Check distribution across schema phases
# display(df_target.groupBy("_schema_phase").count())